In [1]:
import gzip
import csv
import json
import pandas as pd
import os

In [2]:
# --- Uncompressing ---

# 1.1 Uncompress images.csv.gz
images_csv_path = "abo-images-small/images/metadata/images.csv.gz"
images_csv_uncompressed_path = "abo-images-small/images/metadata/images.csv"

with gzip.open(images_csv_path, 'rt') as f_in:
    with open(images_csv_uncompressed_path, 'w', newline='') as f_out:  # newline='' prevents extra empty rows
        csv_reader = csv.reader(f_in)
        csv_writer = csv.writer(f_out)
        for row in csv_reader:
            csv_writer.writerow(row)

print(f"Uncompressed: {images_csv_path} to {images_csv_uncompressed_path}")


# 1.2 Uncompress listings JSON files
listings_dir = "abo-listings/listings/metadata"
uncompressed_listings = []

for filename in os.listdir(listings_dir):
    if filename.endswith(".json.gz"):
        compressed_path = os.path.join(listings_dir, filename)
        uncompressed_path = os.path.join(listings_dir, filename[:-3])  # Remove the .gz extension
        
        with gzip.open(compressed_path, 'rt', encoding='utf-8') as f_in: # Specify encoding for potential Unicode chars
            with open(uncompressed_path, 'w', encoding='utf-8') as f_out:
                for line in f_in:
                    f_out.write(line)
        
        print(f"Uncompressed: {compressed_path} to {uncompressed_path}")
        uncompressed_listings.append(uncompressed_path) # Keep track of uncompressed JSON paths


Uncompressed: abo-images-small/images/metadata/images.csv.gz to abo-images-small/images/metadata/images.csv
Uncompressed: abo-listings/listings/metadata/listings_b.json.gz to abo-listings/listings/metadata/listings_b.json
Uncompressed: abo-listings/listings/metadata/listings_2.json.gz to abo-listings/listings/metadata/listings_2.json
Uncompressed: abo-listings/listings/metadata/listings_c.json.gz to abo-listings/listings/metadata/listings_c.json
Uncompressed: abo-listings/listings/metadata/listings_3.json.gz to abo-listings/listings/metadata/listings_3.json
Uncompressed: abo-listings/listings/metadata/listings_a.json.gz to abo-listings/listings/metadata/listings_a.json
Uncompressed: abo-listings/listings/metadata/listings_1.json.gz to abo-listings/listings/metadata/listings_1.json
Uncompressed: abo-listings/listings/metadata/listings_8.json.gz to abo-listings/listings/metadata/listings_8.json
Uncompressed: abo-listings/listings/metadata/listings_9.json.gz to abo-listings/listings/metad

In [ ]:
# Load the balanced dataset
balanced_df = pd.read_csv("./csv_files/balanced_part2_18000.csv")

# Function to clean text
def clean_text(text):
    if pd.isna(text):
        return None
    return str(text).lower().strip()

# List of columns to clean
columns_to_clean = [
    'main_image_id', 'image_path', 'color', 'model_name', 
    'product_type', 'style', 'material', 'fabric_type', 
    'pattern', 'item_shape'
]

# Create a clean copy of the DataFrame
clean_df = balanced_df.copy()

# Clean each column
for col in columns_to_clean:
    if col in clean_df.columns:
        clean_df[col] = clean_df[col].apply(clean_text)

# Remove rows where essential fields are missing
essential_columns = ['product_type', 'image_path']
clean_df = clean_df.dropna(subset=essential_columns)

# Filter out non-English or unclear values
clean_df = clean_df[
    # Filter out rows where color is not in English
    (~clean_df['color'].str.contains('|'.join(['[^\x00-\x7F]', 'nan']), na=False)) &
    # Filter out rows where material contains non-English characters
    (~clean_df['material'].str.contains('[^\x00-\x7F]', na=False))
]

# Standardize some common values
clean_df['color'] = clean_df['color'].replace({
    'multi-colored': 'multicolor',
    'multi colored': 'multicolor',
    'others': 'other'
})

# Reset index after filtering
clean_df = clean_df.reset_index(drop=True)

print(f"Original dataset size: {len(balanced_df)}")
print(f"Clean dataset size: {len(clean_df)}")
print("\nSample of cleaned data:")
print(clean_df[columns_to_clean].head())

# Save cleaned dataset
clean_df.to_csv("cleaned_balanced_dataset.csv", index=False)
print("\nCleaned dataset saved to cleaned_balanced_dataset.csv")

# Print value counts for important columns to verify cleaning
print("\nUnique values in cleaned columns:")
for col in ['color', 'product_type', 'material']:
    if col in clean_df.columns:
        print(f"\n{col.upper()} value counts:")
        print(clean_df[col].value_counts().head())

Original dataset size: 18000
Clean dataset size: 15824

Sample of cleaned data:
  main_image_id       image_path       color                      model_name  \
0   61b5ls+ozcl  5e/5e3161f1.jpg  multicolor                 xiaomi redmi 8a   
1   81ka6yspzrl  82/82b18223.jpg       other  motorola moto e 1st generation   
2   71pqxn+uzkl  0c/0c0d43e2.jpg       other                       vivo y21l   
3   71zyq0zpjol  ba/ba2bd013.jpg  multicolor                       vivo y51l   
4   71rzxh7+t2l  08/08a09c5d.jpg       other                         oppo f5   

          product_type style material fabric_type pattern item_shape  
0  cellular_phone_case  None  plastic        None    None       None  
1  cellular_phone_case  None     None        None    None       None  
2  cellular_phone_case  None     None        None    None       None  
3  cellular_phone_case  None  silicon        None    None       None  
4  cellular_phone_case  None     None        None    None       None  

Cleaned data

In [ ]:
# %pip install ollama
import ollama
import pandas as pd

def generate_qa(image_path, product_info, model_name="llava:13b"):
    """Generates a question-answer pair based on product type."""
    
    # First, check if this is a phone case or other product
    is_phone_case = "CELLULAR_PHONE_CASE" in product_info
    
    base_prompt = f"""
    You are analyzing product images. The image resolution is 256x256 pixels, so focus on clearly visible features.
    
    Product Information:
    {product_info}
    
    Image Path: {image_path}
    """

    if is_phone_case:
        prompt = base_prompt + """
    **Phone Case Analysis Task:**
    1. Look at the phone case and determine:
       - If there is a visible design/pattern/character/object printed on it
       - The color or transparency of the case
       - The type of case (clear, solid, patterned, printed)
       - Any distinctive material features (silicone, plastic, etc.)

    **Question Types for Phone Cases:**
    1. "What design is printed on this phone case?" (if visible design exists)
    2. "What type of phone case is this?" (clear/solid/patterned)
    3. "What color is this phone case?" (if colored)
    4. "What material is this phone case made of?" (if material is distinctive)
    """
    else:
        prompt = base_prompt + """
    **General Product Analysis Task:**
    1. Look at the product and identify:
       - The main visible object/product category
       - Prominent colors
       - Materials or textures
       - Basic shape or form
       - Notable features or attributes

    **Question Types for General Products:**
    1. "What type of product is this?" (basic category)
    2. "What color is this [product]?" (main color)
    3. "What material is this [product] made of?" (if material is visible)
    4. "What is the shape of this [product]?" (if shape is distinctive)
    """

    prompt += """
    **Requirements:**
    1. Generate ONE question that:
       - Is clearly answerable from both image and metadata
       - Has a single-word answer only
    2. Focus on obvious visual features
    3. Avoid questions about:
       - Small text or details
       - Measurements or dimensions
       - Subjective qualities

    **Output Format (STRICT):**
    Question: <single clear question about visible feature/objects>
    Answer: <single word only>

    **Example Good Outputs:**
    For Phone Case:
    Question: What design is printed on this phone case?
    Answer: flowers

    For Other Product:
    Question: What color is this backpack?
    Answer: blue
    """

    try:
        response = ollama.chat(
            model=model_name,
            messages=[
                {
                    'role': 'user',
                    'content': prompt,
                    'images': [image_path],
                },
            ]
        )
        # Parse the response to extract the question and answer
        text = response['message']['content']
        question = text.split("Question:")[1].split("Answer:")[0].strip()
        answer = text.split("Answer:")[1].strip()
        return question, answer
    except Exception as e:
        print(f"Error generating QA: {e}")
        return None, None

# ollama_model = "gemma3:latest"
ollama_model = "llava:7b"

# Create lists to store the generated Q&A
questions = []
answers = []
paths = []

total_images = len(clean_df)
processed_count = 0
save_interval = 750 #change accordingly
checkpoint_num = 0

# Iterate through the DataFrame
for index, row in clean_df.iterrows():
    image_path = os.path.join("abo-images-small/images/small", row['image_path'])
    # product_info = f"Name: {row['english_item_name']}, Brand: {row['english_brand']}, Color: {row['english_color']}"
    product_info_parts = []
    if pd.notna(row['model_name']):
        product_info_parts.append(f"Model: {row['model_name']}")
    if pd.notna(row['color']):
        product_info_parts.append(f"Color: {row['color']}")
    if pd.notna(row['product_type']):
        product_info_parts.append(f"Type: {row['product_type']}")
    if pd.notna(row['material']):
        product_info_parts.append(f"Material: {row['material']}")
    if pd.notna(row['style']):
        product_info_parts.append(f"Style: {row['style']}")
    if pd.notna(row['pattern']):
        product_info_parts.append(f"Pattern: {row['pattern']}")
    if pd.notna(row['item_shape']):
        product_info_parts.append(f"Pattern: {row['item_shape']}")
    
    product_info = ", ".join(product_info_parts)

    question, answer = generate_qa(image_path, product_info, ollama_model)

    if question and answer:
        questions.append(question)
        answers.append(answer)
        paths.append(row['image_path'])

        if len(paths) % save_interval == 0:
                checkpoint_num += 1
                temp_df = pd.DataFrame({
                    'path': paths,
                    'generated_question': questions,
                    'generated_answer': answers
                })
                
                # Save checkpoint
                checkpoint_file = f"vqa_dataset_checkpoint_{checkpoint_num}.csv"
                temp_df.to_csv(checkpoint_file, index=False)
                print(f"\nCheckpoint {checkpoint_num} saved: {len(paths)} Q&A pairs written to {checkpoint_file}")

    processed_count += 1
    if processed_count % 10 == 0:
        print(f"Processed {processed_count}/{total_images} images ({(processed_count/total_images)*100:.2f}%)")

outputdf = pd.DataFrame({
    'path': paths,
    'generated_question': questions,
    'generated_answer': answers
})

outputdf.to_csv("vqa_dataset_final.csv", index=False)
print(f"\nFinal dataset saved: {len(paths)} total Q&A pairs")
print("\nSample of generated Q&A pairs:")
print(outputdf[['path', 'generated_question', 'generated_answer']].head())

Error generating QA: list index out of range

Sample of generated Q&A pairs:
              path                          generated_question  \
0  38/388cbc26.jpg       What material is the planter made of?   
1  70/7072a118.jpg            What type of phone case is this?   
2  90/9068f2a5.jpg             What type of towel set is this?   
3  00/001d4b57.jpg  What design is printed on this phone case?   
4  a7/a7d14e8f.jpg               What type of product is this?   

        generated_answer  
0                  brass  
1  Motorola moto g6 plus  
2                8-piece  
3               diamonds  
4    cellular_phone_case  


In [ ]:
# %pip install google-generativeai pillow tenacity

import google.generativeai as genai
from PIL import Image
import pandas as pd
import os
import time
from random import uniform
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

# Configure multiple API keys
API_KEYS = [
    'Api1',
    'Api2',
    'Api3',
]

API_USAGE_TRACKER = {i: 0 for i in range(len(API_KEYS))}
current_api_key_index = 0
api_request_count = 0
MAX_REQUESTS_PER_KEY = 1450  # Conservative limit to avoid hitting quota

# Track processed images
last_processed_index = 9000  # Initialize to 0 or load from checkpoint
checkpoint_file = "processing_checkpoint.txt"

def print_progress_report():
    """Print detailed processing statistics"""
    total_processed = len(results)
    current_batch = (last_processed_index // batch_size) + 1
    total_batches = (len(clean_df) + batch_size - 1) // batch_size
    
    print("\n" + "="*50)
    print(f"{'Processing Progress':^50}")
    print("="*50)
    print(f"Current API Key: {current_api_key_index + 1}/{len(API_KEYS)}")
    print(f"Requests on this key: {api_request_count}/{MAX_REQUESTS_PER_KEY}")
    print(f"\nTotal Images Processed: {last_processed_index}/{len(clean_df)}")
    print(f"Current Batch: {current_batch}/{total_batches}")
    print(f"Successful Q&A Pairs: {total_processed}")
    print(f"\nAPI Key Usage Summary:")
    for key_num, count in API_USAGE_TRACKER.items():
        print(f"Key {key_num + 1}: {count} requests")
    print("="*50 + "\n")


def load_checkpoint():
    """Load the last processed index from checkpoint file"""
    global last_processed_index
    try:
        with open(checkpoint_file, 'r') as f:
            last_processed_index = int(f.read())
        print(f"Resuming from index {last_processed_index}")
    except FileNotFoundError:
        last_processed_index = 9000  #change index to start
        print(f"Starting from manual override index {last_processed_index}")

def save_checkpoint(index):
    """Save the current processing index to checkpoint file"""
    with open(checkpoint_file, 'w') as f:
        f.write(str(index))

def switch_to_next_api_key():
    """Switch to the next available API key"""
    global current_api_key_index, api_request_count
    current_api_key_index = (current_api_key_index + 1) % len(API_KEYS)
    api_request_count = 0
    genai.configure(api_key=API_KEYS[current_api_key_index])
    print(f"\nSwitching to API key {current_api_key_index + 1}")
    time.sleep(5)  # Longer pause when switching keys to avoid rate limits

# Initialize with first API key
genai.configure(api_key=API_KEYS[current_api_key_index])
gemini_model = genai.GenerativeModel('gemini-1.5-flash')  # Updated to newer model

@retry(
    wait=wait_exponential(multiplier=1, min=4, max=60),
    stop=stop_after_attempt(3),
    retry=retry_if_exception_type(Exception)
)
def generate_qa_gemini(image_path, product_info):
    """Generates a question-answer pair using Gemini Vision API with rate limiting."""
    global api_request_count, API_USAGE_TRACKER
    
    prompt = f"""Analyze this product image carefully. The image resolution is 256x256 pixels.

Product Information:
{product_info}

Task:
1. Look at the product and identify:
   - The main visible object/product category
   - Prominent colors
   - Materials or textures
   - Basic shape or form
   - Notable features or attributes

Requirements:
1. Generate ONE question that:
   - Must be clearly answerable from the image
   - Must have a single-word answer only
   - Must focus on obvious visual features
2. Avoid questions about:
   - Small text or details
   - Measurements or dimensions
   - Subjective qualities
   - Brand names unless clearly visible

Output Format (STRICT):
Question: <single clear question about visible feature>
Answer: <single word only>

Example Good Outputs:
Question: What color is this shirt?
Answer: blue

Question: What material is this table made of?
Answer: wood"""
    
    try:
        if api_request_count >= MAX_REQUESTS_PER_KEY:
            if current_api_key_index == len(API_KEYS) - 1:
                print("\nAll API keys exhausted. Saving progress...")
                return None, None, True
            switch_to_next_api_key()
        
        time.sleep(uniform(3, 5))
        
        try:
            image = Image.open(image_path)
        except Exception as e:
            print(f"Error opening image {image_path}: {e}")
            return None, None, False

        response = gemini_model.generate_content([prompt, image])
        api_request_count += 1
        API_USAGE_TRACKER[current_api_key_index] += 1


        response = gemini_model.generate_content([prompt, image])
        api_request_count += 1
        
        # Parse response with improved error handling
        text = response.text
        try:
            question = text.split("Question:")[1].split("Answer:")[0].strip()
            answer = text.split("Answer:")[1].strip().split()[0]  # Take first word only
            return question, answer, False
        except IndexError:
            print(f"Failed to parse response: {text}")
            return None, None, False
    
    except Exception as e:
        error_msg = str(e).lower()
        if "quota" in error_msg or "exceeded" in error_msg:
            print(f"\nQuota exceeded for API key {current_api_key_index + 1}")
            if current_api_key_index == len(API_KEYS) - 1:
                print("All API keys exhausted. Saving progress...")
                return None, None, True
            switch_to_next_api_key()
            return None, None, False
        else:
            print(f"Gemini Error for {image_path}: {e}")
            return None, None, False

# Load your dataframe
# clean_df = pd.read_csv("your_dataset.csv")  # Replace with your actual dataframe
load_checkpoint()  # Load last processed index

# Create lists to store results
results = []
batch_size = 50
save_interval = 500

# Main processing loop
for index, row in clean_df.iloc[last_processed_index:].iterrows():
    image_path = os.path.join("abo-images-small/images/small", row['image_path'])
    
    # Construct product info from available fields
    product_info_parts = []
    for field in ['model_name', 'color', 'product_type', 'material', 'style', 'pattern', 'item_shape']:
        if pd.notna(row[field]):
            product_info_parts.append(f"{field.capitalize()}: {row[field]}")
    product_info = ", ".join(product_info_parts)

    question, answer, apis_exhausted = generate_qa_gemini(image_path, product_info)

    if (index + 1) % 10 == 0 or (index + 1) % batch_size == 0:
        print_progress_report()
        # Additional batch completion message
        if (index + 1) % batch_size == 0:
            print(f"Completed batch {(index + 1) // batch_size}")
            print(f"Last processed image: {row['image_path']}")

    if question and answer:
        results.append({
            'path': row['image_path'],
            'question': question,
            'answer': answer,
            'product_info': product_info
        })

    # Save progress periodically
    if len(results) > 0 and len(results) % save_interval == 0:
        pd.DataFrame(results).to_csv(f"vqa_progress_{len(results)}.csv", index=False)
        save_checkpoint(index)
        print(f"Saved progress at {len(results)} items")

    # Update last processed index
    last_processed_index = index + 1

    # Pause every batch to avoid rate limits
    if (index + 1) % batch_size == 0:
        time.sleep(30)

# Final save
if len(results) > 0:
    pd.DataFrame(results).to_csv("vqa_dataset_final.csv", index=False)
    save_checkpoint(last_processed_index)
    print(f"\nProcessing completed. Total Q&A pairs: {len(results)}")
else:
    print("\nNo new Q&A pairs generated")

print("\n" + "="*50)
print(f"{'FINAL PROCESSING REPORT':^50}")
print("="*50)
print(f"Total Images Processed: {last_processed_index}")
print(f"Successful Q&A Pairs: {len(results)}")
print(f"Success Rate: {(len(results)/last_processed_index)*100:.2f}%")
print("\nAPI Key Usage Summary:")
for key_num, count in API_USAGE_TRACKER.items():
    print(f"Key {key_num + 1}: {count} requests")
print("="*50)

In [23]:
# Create the final QA dataset DataFrame
qa_df = outputdf[['path', 'generated_question', 'generated_answer']].copy()
qa_df = qa_df.dropna(subset=['generated_question', 'generated_answer'])

# Save to CSV
qa_df.to_csv("vqa_dataset.csv", index=False)
print("VQA dataset saved to vqa_dataset.csv")

VQA dataset saved to vqa_dataset.csv
